<a href="https://colab.research.google.com/github/amany2712/project_nlp/blob/preprocessing/project_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('stopwords')
nltk.download('punkt')
url_data = "https://raw.githubusercontent.com/amany2712/project_nlp/refs/heads/main/spam_email_dataset.csv"
df = pd.read_csv(url_data)
df.head(100)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,Schedule Confirmation,budget review - Turn sea understand guess. Pic...,15,105,0,4,0,0,0,jkjzdcs@outlook.com,outlook.com,0.82,0,1,0,7,0,0,0
96,96,Limited Offer!!!,offer cash click now offer offer free Nice ind...,15,90,0,4,1,1,1,zamhcmh@freemoney.biz,freemoney.biz,0.32,21,2,0,6,1,1,1
97,97,Weekly Report,budget review - Seat whose professor machine t...,19,133,0,6,1,1,1,rczgcbu@company.com,company.com,0.56,19,3,0,16,0,1,0
98,98,🔥WIN BIG NOW!!,guarantee limited win urgent win click now urg...,14,90,0,0,0,0,0,btqnwew@chealdealz.xyz,chealdealz.xyz,0.40,6,0,0,3,1,1,1


In [19]:
df.isnull().sum()

,0
email_id,0
subject,0
email_text,0
num_words,0
num_characters,0
num_exclamation_marks,0
num_links,0
has_suspicious_link,0
num_attachments,0
has_attachment,0


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   email_id                 10000 non-null  int64  
 1   subject                  10000 non-null  object 
 2   email_text               10000 non-null  object 
 3   num_words                10000 non-null  int64  
 4   num_characters           10000 non-null  int64  
 5   num_exclamation_marks    10000 non-null  int64  
 6   num_links                10000 non-null  int64  
 7   has_suspicious_link      10000 non-null  int64  
 8   num_attachments          10000 non-null  int64  
 9   has_attachment           10000 non-null  int64  
 10  sender_email             10000 non-null  object 
 11  sender_domain            10000 non-null  object 
 12  sender_reputation_score  10000 non-null  float64
 13  email_hour               10000 non-null  int64  
 14  email_day_of_week      

In [22]:
df=df.drop(columns=['email_id', 'sender_email'])

In [32]:
import re                       # تنظيف النص
import nltk                     # مكتبة اللغة
from nltk.corpus import stopwords  # كلمات زي "the, is, and"
from nltk.stem import PorterStemmer # رجّع الكلمة لأصلها
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
ps= PorterStemmer()
stop = set(stopwords.words('english'))
def clean_text(text):
    if pd.isna(text): return ''
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [ps.stem(w) for w in text.split() if w not in stop]
    return ' '.join(tokens)

df['combined_text'] = (df['subject'].fillna('') + ' ' +
                        df['email_text'].fillna('')).apply(clean_text)

print("✅ خلص!")
print(df['combined_text'].head(3))

✅ خلص!
0    weekli report budget review statement claim wo...
1    project updat team sync presid seri today alre...
2    win big win free urgent offer limit limit urge...
Name: combined_text, dtype: object


In [33]:
from sklearn.model_selection import train_test_split

num_cols = ['num_words','num_characters','num_exclamation_marks',
            'num_links','has_suspicious_link','num_attachments',
            'has_attachment','sender_reputation_score','email_hour',
            'email_day_of_week','is_weekend','num_recipients',
            'contains_money_terms','contains_urgency_terms']

X_tr_txt, X_te_txt, y_train, y_test = train_test_split(
    df['combined_text'], df['label'],
    test_size=0.2, random_state=42
)

X_tr_num = df.loc[X_tr_txt.index, num_cols]
X_te_num = df.loc[X_te_txt.index, num_cols]

print("Train size:", len(X_tr_txt))
print("Test size: ", len(X_te_txt))

Train size: 8000
Test size:  2000


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tr_tfidf = tfidf.fit_transform(X_tr_txt)
X_te_tfidf = tfidf.transform(X_te_txt)

# Scaler
scaler = StandardScaler()
X_tr_num_sc = csr_matrix(scaler.fit_transform(X_tr_num))
X_te_num_sc = csr_matrix(scaler.transform(X_te_num))

# Merge
X_train = hstack([X_tr_tfidf, X_tr_num_sc])
X_test  = hstack([X_te_tfidf, X_te_num_sc])

print("✅ جاهز!")
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

✅ جاهز!
X_train shape: (8000, 868)
X_test shape:  (2000, 868)
